In [4]:
import os

# Search for any .pt files saved by YOLO
print("Searching for saved YOLO weights...\n")
search_dirs = [
    r'C:\Users\Super\anaconda_projects\aerial',
    r'C:\Users\Super',
    os.getcwd()   # current working directory of your notebook
]

for base in search_dirs:
    if not os.path.exists(base):
        continue
    for dirpath, dirnames, filenames in os.walk(base):
        for f in filenames:
            if f.endswith('.pt'):
                print(f"  Found: {os.path.join(dirpath, f)}")
        # Don't go too deep
        if dirpath.count(os.sep) - base.count(os.sep) > 5:
            break

Searching for saved YOLO weights...

  Found: C:\Users\Super\anaconda_projects\aerial\yolov8n.pt
  Found: C:\Users\Super\anaconda_projects\aerial\runs\detect\yolo_runs\bird_drone_yolov8\weights\best.pt
  Found: C:\Users\Super\anaconda_projects\aerial\runs\detect\yolo_runs\bird_drone_yolov8\weights\last.pt
  Found: C:\Users\Super\anaconda_projects\aerial\yolov8n.pt
  Found: C:\Users\Super\anaconda_projects\aerial\runs\detect\yolo_runs\bird_drone_yolov8\weights\best.pt
  Found: C:\Users\Super\anaconda_projects\aerial\runs\detect\yolo_runs\bird_drone_yolov8\weights\last.pt


In [5]:
from ultralytics import YOLO

# Correct path
YOLO_BEST = r'C:\Users\Super\anaconda_projects\aerial\runs\detect\yolo_runs\bird_drone_yolov8\weights\best.pt'

best_yolo = YOLO(YOLO_BEST)
print("✅ Best YOLO model loaded")

# Validate
metrics = best_yolo.val()
print(f"\n  mAP@0.5      : {metrics.box.map50:.4f}")
print(f"  mAP@0.5:0.95 : {metrics.box.map:.4f}")
print(f"  Precision    : {metrics.box.mp:.4f}")
print(f"  Recall       : {metrics.box.mr:.4f}")

✅ Best YOLO model loaded
Ultralytics 8.4.33  Python-3.13.9 torch-2.11.0+cpu CPU (Intel Core i5-8250U 1.60GHz)
Model summary (fused): 73 layers, 3,006,038 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access  (ping: 0.70.7 ms, read: 14.65.6 MB/s, size: 17.3 KB)
val: Scanning D:\Tejashri\Aeriel\Object_detection_dataset\valid\labels.cache... 448 images, 6 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 448/448 13.2Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 28/28 3.9s/it 1:493.5sss
                   all        448        663      0.691      0.633      0.686      0.393
                  bird        217        414      0.631      0.418      0.505      0.252
                 drone        225        249       0.75      0.847      0.867      0.534
Speed: 2.3ms preprocess, 215.8ms inference, 0.0ms loss, 3.3ms postprocess per image
Results saved to C:\Users\Super\anaconda_projects\aerial\runs\detect\val

  mAP@0.5      :

In [6]:
import shutil, os

DEPLOY_DIR = r'C:\Users\Super\anaconda_projects\aerial\streamlit_app'
os.makedirs(DEPLOY_DIR, exist_ok=True)

shutil.copy(YOLO_BEST, f'{DEPLOY_DIR}\\yolo_best.pt')
print(f"✅ YOLO model saved to: {DEPLOY_DIR}\\yolo_best.pt")

✅ YOLO model saved to: C:\Users\Super\anaconda_projects\aerial\streamlit_app\yolo_best.pt


In [7]:
DETECT_PATH = r'D:\Tejashri\Aeriel\Object_detection_dataset'

results = best_yolo.predict(
    source=f'{DETECT_PATH}\\test\\images',
    conf=0.25,
    iou=0.45,
    save=True,
    project=r'C:\Users\Super\anaconda_projects\aerial\yolo_output',
    name='predictions',
    verbose=False
)

print(f"✅ Inference done on {len(results)} images")

birds, drones = 0, 0
for r in results:
    for cls in r.boxes.cls.tolist():
        if int(cls) == 0: birds  += 1
        else:             drones += 1

print(f"   Birds  detected : {birds}")
print(f"   Drones detected : {drones}")
print(f"   Saved to: C:\\Users\\Super\\anaconda_projects\\aerial\\yolo_output\\predictions\\")

Results saved to C:\Users\Super\anaconda_projects\aerial\yolo_output\predictions
✅ Inference done on 224 images
   Birds  detected : 148
   Drones detected : 108
   Saved to: C:\Users\Super\anaconda_projects\aerial\yolo_output\predictions\
